In [ ]:
# Cell 1: Bootstrap
import sys
from pathlib import Path

nb_dir = Path.cwd() if '__file__' not in dir() else Path(__file__).parent
sys.path.insert(0, str(nb_dir))
from _bootstrap import bootstrap
bootstrap()

## PRD-102: Economic Sentiment Validation (Beige Book + FinBERT)

**Objective:** Empirical validation of FinBERT-FOMC applied to Beige Book economic text.

FinBERT was removed from the FOMC Rhetoric ensemble (20% accuracy on policy stance).
Here we validate the **correct** use case: economic condition sentiment where
`Positive -> economy strong` and `Negative -> economy weak` (direct mapping, no inversion).

**Key questions:**
1. Do recessions (2008, 2020) show clear drops in sentiment?
2. Does the national vs. district divergence reveal useful patterns?
3. Which districts are most/least volatile?
4. Does sentiment correlate with real rate differential per regime?

In [ ]:
# Cell 3: Run pipeline (limited to 2016+ for initial validation speed)
from macro_context_reader.economic_sentiment.pipeline import run_full_pipeline

df = run_full_pipeline(start_year=2016)
print(f"Publications scored: {len(df)}")
df.head(10)

In [ ]:
# Cell 4: Distribution plots
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if 'national_score' in df.columns:
    df['national_score'].dropna().hist(ax=axes[0], bins=20, edgecolor='black')
    axes[0].set_title('National Score Distribution')
    axes[0].set_xlabel('Sentiment Score [-1, +1]')

df['district_weighted_score'].hist(ax=axes[1], bins=20, edgecolor='black', color='steelblue')
axes[1].set_title('District Weighted Score Distribution')
axes[1].set_xlabel('Sentiment Score [-1, +1]')

if 'national_district_divergence' in df.columns:
    df['national_district_divergence'].dropna().hist(ax=axes[2], bins=20, edgecolor='black', color='coral')
    axes[2].set_title('National-District Divergence')
    axes[2].set_xlabel('Divergence (national - district)')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Time series — national vs district weighted
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(14, 5))

dates = pd.to_datetime(df['publication_date'])

if 'national_score' in df.columns:
    ax.plot(dates, df['national_score'], label='National Summary', marker='o', markersize=3, alpha=0.7)

ax.plot(dates, df['district_weighted_score'], label='District Weighted', marker='s', markersize=3, alpha=0.7)

# NBER recession shading
recession_periods = [
    ('2007-12-01', '2009-06-30'),  # Great Recession
    ('2020-02-01', '2020-04-30'),  # COVID
]
for start, end in recession_periods:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.2, color='gray', label='_nolegend_')

ax.set_ylabel('Sentiment Score [-1, +1]')
ax.set_title('Beige Book Economic Sentiment Over Time')
ax.legend()
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

# Verify COVID recession visible
covid_mask = (dates >= '2020-01-01') & (dates <= '2020-12-31')
if covid_mask.any():
    covid_min = df.loc[covid_mask, 'district_weighted_score'].min()
    print(f"COVID min district sentiment: {covid_min:.3f}")
    print(f"Overall mean: {df['district_weighted_score'].mean():.3f}")

In [ ]:
# Cell 6: Cross-district heatmap
import numpy as np
import matplotlib.pyplot as plt

district_cols = [c for c in df.columns if c.endswith('_score') and c not in (
    'national_score', 'district_weighted_score'
)]

if district_cols:
    district_df = df[['publication_date'] + district_cols].copy()
    district_df = district_df.set_index('publication_date')
    district_df.columns = [c.replace('_score', '').replace('_', ' ') for c in district_df.columns]
    
    fig, ax = plt.subplots(figsize=(16, 8))
    im = ax.imshow(district_df.T.values, aspect='auto', cmap='RdYlGn', vmin=-0.5, vmax=0.5)
    ax.set_yticks(range(len(district_df.columns)))
    ax.set_yticklabels(district_df.columns)
    
    # X-axis: show every ~10th date
    step = max(1, len(district_df) // 15)
    tick_positions = range(0, len(district_df), step)
    tick_labels = [str(district_df.index[i])[:10] for i in tick_positions]
    ax.set_xticks(list(tick_positions))
    ax.set_xticklabels(tick_labels, rotation=45, ha='right')
    
    plt.colorbar(im, label='Sentiment Score')
    ax.set_title('Beige Book Sentiment by District Over Time')
    plt.tight_layout()
    plt.show()
    
    # Volatility ranking
    vol = district_df.std().sort_values(ascending=False)
    print("\nDistrict Volatility (std of sentiment score):")
    for d, v in vol.items():
        print(f"  {d:<20} {v:.4f}")
else:
    print("No district columns found — check pipeline output.")

In [ ]:
# Cell 7: Cross-layer validation with regime + real_rate_diff
from scipy.stats import pearsonr
import pandas as pd

# Attempt to load regime and real rate diff data
try:
    regime_path = 'data/regime/regime_labels.parquet'
    regime_df = pd.read_parquet(regime_path)
    print(f"Regime data: {len(regime_df)} rows")
except Exception as e:
    print(f"Regime data not available: {e}")
    regime_df = None

try:
    rrd_path = 'data/market_pricing/real_rate_diff.parquet'
    rrd_df = pd.read_parquet(rrd_path)
    print(f"Real rate diff data: {len(rrd_df)} rows")
except Exception as e:
    print(f"Real rate diff data not available: {e}")
    rrd_df = None

if regime_df is not None and rrd_df is not None:
    # Merge on nearest date
    sentiment = df[['publication_date', 'district_weighted_score']].copy()
    sentiment['date'] = pd.to_datetime(sentiment['publication_date'])
    sentiment = sentiment.set_index('date').sort_index()

    # Align regime labels
    if 'date' in regime_df.columns:
        regime_df['date'] = pd.to_datetime(regime_df['date'])
        regime_df = regime_df.set_index('date').sort_index()
    
    if 'date' in rrd_df.columns:
        rrd_df['date'] = pd.to_datetime(rrd_df['date'])
        rrd_df = rrd_df.set_index('date').sort_index()

    merged = sentiment.copy()
    merged = merged.join(regime_df[['hmm_label']], how='left')
    merged = merged.join(rrd_df[['real_rate_diff']], how='left')
    merged = merged.dropna(subset=['hmm_label', 'real_rate_diff'])

    print(f"\nMerged data: {len(merged)} rows")
    print("\nCorrelations per regime:")
    for regime_label in sorted(merged['hmm_label'].unique()):
        sub = merged[merged['hmm_label'] == regime_label]
        if len(sub) >= 5:
            r, p = pearsonr(sub['district_weighted_score'], sub['real_rate_diff'])
            print(f"  {regime_label}: r={r:.3f}, p={p:.4f}, N={len(sub)}")
        else:
            print(f"  {regime_label}: N={len(sub)} (too few observations)")
else:
    print("\nCross-layer validation skipped: missing regime or real_rate_diff data.")
    print("Run PRD-050 (regime) and PRD-200 (market pricing) first.")

## VERDICT

**Validation criteria:**

1. **Recession detection:** If COVID 2020 shows clear drop in sentiment -> FinBERT works on Beige Book
2. **National vs. district divergence:** If patterns visible -> validates dual-perspective hypothesis
3. **Cross-layer correlation:** If sentiment correlates with real_rate_diff per regime -> integrate into PRD-300

**Decision:**
- [ ] PASS: Integrate `district_weighted_score` into PRD-300 as supplementary feature for `surprise_score`
- [ ] FAIL: FinBERT insufficient even for economic sentiment. Consider alternative models.

**Notes:**
- Fill in after running all cells above
- Record specific r-values and p-values from Cell 7